<a href="https://colab.research.google.com/github/LusciousRCChigwena/Luscious-R-C-Chigwena-R2420728/blob/main/Luscious_Chigwena_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Application Time Series Assignment 3: AAPL Regime Change Model

**Chosen model:** Detecting a regime change  
**Dataset:** Apple Inc. (AAPL) daily adjusted closing prices from Yahoo Finance  
**Source page supplied for the data:** https://finance.yahoo.com/quote/AAPL/history/  
**Frequency:** Daily market observations  
**Units:** U.S. dollars for prices; continuously compounded daily returns for the model

The Yahoo Finance history table is loaded dynamically in a browser. This notebook therefore uses Yahoo Finance's chart JSON endpoint for the same AAPL historical price series while keeping the supplied Yahoo Finance history page as the data source.

## Definition

Let \(P_t\) be the adjusted closing price of AAPL on trading day \(t\). The continuously compounded daily return is

\[
r_t = 100 \times \log\left(\frac{P_t}{P_{t-1}}\right).
\]

A two-regime return model assumes returns are generated by an unobserved state \(S_t \in \{0,1\}\):

\[
r_t = \mu_{S_t} + \sigma_{S_t}\epsilon_t, \qquad \epsilon_t \sim N(0,1).
\]

The latent state can be interpreted as a market condition, usually a lower-volatility regime and a higher-volatility regime. After estimating the state sequence, the regime persistence is summarized by

\[
p_{ij}=Pr(S_t=j \mid S_{t-1}=i), \qquad i,j \in \{0,1\}.
\]

The calibrated parameters are the regime means \(\mu_j\), volatilities \(\sigma_j\), mixture weights \(\pi_j\), and empirical transition probabilities \(p_{ij}\).

## Description

The model separates AAPL returns into periods with different average returns and volatility. A regime change is detected when the probability of the high-volatility state rises and stays elevated, showing that the return-generating process is behaving differently from normal conditions.

In [ ]:
import json
import math
import urllib.request
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.mixture import GaussianMixture

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
SYMBOL = "AAPL"
SOURCE_PAGE_URL = "https://finance.yahoo.com/quote/AAPL/history/"
YAHOO_CHART_URL = (
    "https://query1.finance.yahoo.com/v8/finance/chart/"
    f"{SYMBOL}?range=5y&interval=1d&events=history&includeAdjustedClose=true"
)


def fetch_yahoo_history(url: str) -> pd.DataFrame:
    """Fetch Yahoo Finance daily chart data and return a clean price table."""
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=30) as response:
        payload = json.load(response)

    chart = payload["chart"]["result"][0]
    quote = chart["indicators"]["quote"][0]
    adjclose = chart["indicators"].get("adjclose", [{}])[0].get("adjclose")

    data = pd.DataFrame({
        "date": pd.to_datetime(chart["timestamp"], unit="s", utc=True).tz_convert(None).date,
        "open": quote["open"],
        "high": quote["high"],
        "low": quote["low"],
        "close": quote["close"],
        "volume": quote["volume"],
        "adj_close": adjclose if adjclose is not None else quote["close"],
    })
    data["date"] = pd.to_datetime(data["date"])
    data = data.dropna(subset=["adj_close"]).sort_values("date").reset_index(drop=True)
    return data


prices = fetch_yahoo_history(YAHOO_CHART_URL)
prices["log_return_pct"] = 100 * np.log(prices["adj_close"] / prices["adj_close"].shift(1))
returns = prices.dropna(subset=["log_return_pct"]).copy()

print(f"Source page: {SOURCE_PAGE_URL}")
print(f"Downloaded rows: {len(prices):,}")
print(f"Date range: {prices['date'].min().date()} to {prices['date'].max().date()}")
prices.head()

## Demonstration: Prepare the Data

Adjusted closing prices are used because they account for corporate actions such as splits and dividends. The model is applied to daily log returns rather than prices because regime changes in liquid equities often appear as shifts in volatility and downside risk rather than as stable changes in the price level.

In [ ]:
summary = returns["log_return_pct"].describe().to_frame("AAPL daily log return (%)")
summary.loc["skewness"] = returns["log_return_pct"].skew()
summary.loc["excess_kurtosis"] = returns["log_return_pct"].kurtosis()
summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(prices["date"], prices["adj_close"], color="#0f766e", linewidth=1.4)
axes[0].set_title("AAPL Adjusted Closing Price")
axes[0].set_ylabel("Adjusted close ($)")

axes[1].plot(returns["date"], returns["log_return_pct"], color="#334155", linewidth=0.8)
axes[1].axhline(0, color="#111827", linewidth=0.8)
axes[1].set_title("AAPL Daily Log Returns")
axes[1].set_ylabel("Return (%)")
axes[1].set_xlabel("Date")

fig.tight_layout()
plt.show()

## Demonstration: Run the Model

The fitted model uses two Gaussian return regimes. The higher estimated standard deviation is labeled as the stressed or high-volatility regime. This state is the main evidence of regime change because it signals that the same stock is temporarily being priced under a different risk environment.

In [ ]:
X = returns[["log_return_pct"]].to_numpy()

model = GaussianMixture(n_components=2, covariance_type="full", random_state=42, n_init=25)
model.fit(X)

probabilities = model.predict_proba(X)
raw_state = probabilities.argmax(axis=1)
means = model.means_.ravel()
stds = np.sqrt(model.covariances_.ravel())
weights = model.weights_.ravel()

high_vol_raw_state = int(np.argmax(stds))
state_name = np.where(raw_state == high_vol_raw_state, "High volatility", "Low volatility")
high_vol_probability = probabilities[:, high_vol_raw_state]

returns["regime"] = state_name
returns["high_vol_probability"] = high_vol_probability

parameter_table = pd.DataFrame({
    "raw_state": [0, 1],
    "regime_label": ["High volatility" if i == high_vol_raw_state else "Low volatility" for i in [0, 1]],
    "weight": weights,
    "mean_daily_return_pct": means,
    "daily_volatility_pct": stds,
    "annualized_volatility_pct": stds * math.sqrt(252),
}).sort_values("daily_volatility_pct").reset_index(drop=True)

parameter_table

In [ ]:
ordered_states = np.where(returns["regime"].eq("High volatility"), 1, 0)
transition_counts = pd.DataFrame(0, index=["Low volatility", "High volatility"], columns=["Low volatility", "High volatility"])

labels = np.array(["Low volatility", "High volatility"])
for previous, current in zip(ordered_states[:-1], ordered_states[1:]):
    transition_counts.loc[labels[previous], labels[current]] += 1

transition_matrix = transition_counts.div(transition_counts.sum(axis=1), axis=0)
transition_matrix

## Diagram

The following charts show the return series, the inferred high-volatility probability, and the classified regimes. Sustained clusters of high-volatility observations are interpreted as regime-change episodes.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(returns["date"], returns["log_return_pct"], color="#475569", linewidth=0.8)
axes[0].scatter(
    returns.loc[returns["regime"].eq("High volatility"), "date"],
    returns.loc[returns["regime"].eq("High volatility"), "log_return_pct"],
    color="#dc2626",
    s=10,
    label="High volatility regime",
)
axes[0].set_title("Daily Returns with High-Volatility Regime Highlighted")
axes[0].set_ylabel("Return (%)")
axes[0].legend(loc="upper right")

axes[1].plot(returns["date"], returns["high_vol_probability"], color="#7c3aed", linewidth=1.1)
axes[1].axhline(0.5, color="#111827", linestyle="--", linewidth=0.8)
axes[1].set_title("Probability of High-Volatility Regime")
axes[1].set_ylabel("Probability")

rolling_vol = returns.set_index("date")["log_return_pct"].rolling(21).std() * math.sqrt(252)
axes[2].plot(rolling_vol.index, rolling_vol, color="#0f766e", linewidth=1.1)
axes[2].set_title("21-Day Rolling Annualized Volatility")
axes[2].set_ylabel("Annualized volatility (%)")
axes[2].set_xlabel("Date")

fig.tight_layout()
plt.show()

## Diagnosis

Model diagnostics compare the empirical return distribution with the fitted two-regime mixture and examine standardized residuals. A good fit should capture fat tails better than a single normal distribution, but residual clustering can still remain because stock returns often have time-varying volatility.

In [ ]:
grid = np.linspace(returns["log_return_pct"].quantile(0.005), returns["log_return_pct"].quantile(0.995), 500)
density = np.zeros_like(grid)
for weight, mean, std in zip(weights, means, stds):
    density += weight * norm.pdf(grid, mean, std)

single_mean = returns["log_return_pct"].mean()
single_std = returns["log_return_pct"].std()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(returns["log_return_pct"], bins=60, density=True, alpha=0.35, color="#64748b", label="Observed returns")
ax.plot(grid, density, color="#dc2626", linewidth=2, label="Two-regime fitted density")
ax.plot(grid, norm.pdf(grid, single_mean, single_std), color="#0f766e", linewidth=1.6, linestyle="--", label="Single normal density")
ax.set_title("Observed AAPL Returns vs Fitted Densities")
ax.set_xlabel("Daily log return (%)")
ax.set_ylabel("Density")
ax.legend()
plt.show()

In [ ]:
state_mean = np.where(returns["regime"].eq("High volatility"), means[high_vol_raw_state], means[1 - high_vol_raw_state])
state_std = np.where(returns["regime"].eq("High volatility"), stds[high_vol_raw_state], stds[1 - high_vol_raw_state])
returns["standardized_residual"] = (returns["log_return_pct"] - state_mean) / state_std

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(returns["date"], returns["standardized_residual"], color="#334155", linewidth=0.8)
axes[0].axhline(0, color="#111827", linewidth=0.8)
axes[0].set_title("Standardized Residuals Over Time")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Residual")

axes[1].hist(returns["standardized_residual"], bins=50, density=True, alpha=0.4, color="#64748b")
x = np.linspace(-4, 4, 300)
axes[1].plot(x, norm.pdf(x), color="#dc2626", linewidth=2)
axes[1].set_title("Residual Distribution vs Standard Normal")
axes[1].set_xlabel("Standardized residual")
axes[1].set_ylabel("Density")

fig.tight_layout()
plt.show()

## Parameter Interpretation

- **Regime mean:** The average daily return in each state. If the high-volatility state has a lower or negative mean, it suggests stress periods are associated with weaker AAPL performance.
- **Regime volatility:** The daily and annualized standard deviation within each state. The high-volatility state measures how much risk increases during turbulent periods.
- **Mixture weight:** The estimated share of observations belonging to each state. A small high-volatility weight means stress periods are less common but potentially important.
- **Transition probabilities:** The diagonal values show persistence. A high high-to-high transition probability means stress regimes tend to continue once they begin.

In [ ]:
latest = returns.iloc[-1]
high_vol_share = returns["regime"].eq("High volatility").mean()

print(f"Latest observation: {latest['date'].date()}")
print(f"Latest AAPL daily return: {latest['log_return_pct']:.2f}%")
print(f"Latest high-volatility probability: {latest['high_vol_probability']:.2%}")
print(f"Share of days classified as high volatility: {high_vol_share:.2%}")
print("\nTransition matrix:")
print(transition_matrix.round(3))

## Damage

The model reveals that AAPL returns are not generated by one stable distribution. Periods of elevated volatility create fat tails, clustered large moves, and changing risk. These issues relate directly to common time-series challenges: non-constant variance, outliers, structural instability, model misspecification, heavy tails, and sensitivity to sample period. A single average return and single volatility estimate can therefore understate risk during stressed market periods.

## Directions

The same model could be refit after shortening the time horizon, winsorizing extreme one-day returns, or adding market-wide explanatory variables such as the S&P 500 return or VIX. However, removing too many extreme observations may hide the exact stress periods the model is meant to detect. For investment use, the better next step is to compare this two-regime model with a model that explicitly includes time dependence in volatility.

## Deployment

An investor or risk manager could run this notebook daily after the market close. If the high-volatility probability rises above a selected threshold, such as 50% or 70%, the portfolio process could trigger a risk review: reduce position size, rebalance exposures, check stop-loss levels, or hedge with index or options exposure. The model should not be used as a standalone buy-or-sell signal. Its practical value is as an early warning system that the current AAPL risk environment differs from normal trading conditions.

## Bibliography

Hamilton, James D. "A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle." *Econometrica*, vol. 57, no. 2, 1989, pp. 357-384.

Pedregosa, Fabian, et al. "Scikit-learn: Machine Learning in Python." *Journal of Machine Learning Research*, vol. 12, 2011, pp. 2825-2830.

Yahoo Finance. "Apple Inc. (AAPL) Stock Historical Prices & Data." *Yahoo Finance*, https://finance.yahoo.com/quote/AAPL/history/. Accessed 7 May 2026.